# 01 - Loading and Visualizing EPR Data

`eprload` is the single entry point for reading Bruker files. It auto-detects the
format and returns a 4-tuple `(x, y, params, filepath)`:

- `x`: the abscissa (magnetic field, time, ...). A single array for 1D data, a
  list of arrays for 2D data.
- `y`: the signal. Real or complex, shape `(N,)` for 1D or `(rows, cols)` for 2D.
- `params`: a dict of acquisition parameters read from the header.
- `filepath`: the resolved path of the file that was loaded.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")  # keep tutorial output readable

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import epyr

DATA = Path("..") / "data"   # example datasets, relative to this notebook
print("EPyR Tools version:", epyr.__version__)

## 1D continuous-wave (CW) spectrum, BES3T format

In [ ]:
x, y, params, filepath = epyr.eprload(DATA / "130406SB_CaWO4_Er_CW_5K_20.DSC",
                                      plot_if_possible=False)

print("x:", type(x).__name__, x.shape, "->", x.min(), "to", x.max())
print("y:", type(y).__name__, y.shape, y.dtype)
print("loaded:", Path(filepath).name)

### Inspecting acquisition parameters

`params` is a plain dict; here are a few entries.

In [ ]:
print(f"{len(params)} parameters in the header. A sample:\n")
for key in list(params)[:10]:
    print(f"  {key:12s} = {params[key]}")

### Plotting

`plot_1d` returns the `(figure, axes)` it draws on, so you can customize further.

In [ ]:
fig, ax = epyr.plot_1d(x, y, params, title="CaWO4:Er - CW EPR at 5 K")
plt.show()

## ESP / WinEPR format

Field-modulated CW spectra are recorded as the first derivative of the absorption.

In [ ]:
x2, y2, params2, _ = epyr.eprload(DATA / "CuSO4_001.par", plot_if_possible=False)
fig, ax = epyr.plot_1d(x2, y2, params2, title="CuSO4 - first-derivative CW spectrum")
plt.show()

## 2D data: an angular rotation map

For 2D datasets `x` is a list of two axes and `y` is a 2D array. Here a single
crystal is rotated through 180 degrees while a field sweep is recorded at each
angle.

In [ ]:
x3, y3, params3, _ = epyr.eprload(DATA / "2014_03_19_MgO_300K_111_fullrotation33dB.par",
                                  plot_if_possible=False)
field, angle = x3[0], x3[1]
print("signal shape (angles x field):", y3.shape)
print("field axis:", field.shape, "  angle axis:", angle.shape)

### Color map and waterfall views

In [ ]:
fig, ax = epyr.plot_2d_map(x3, y3, params3, title="MgO rotation map", cmap="RdBu_r")
plt.show()

In [ ]:
fig, ax = epyr.plot_2d_waterfall(x3, y3, params3, title="MgO rotation - waterfall",
                                 offset_factor=0.4, max_traces=18)
plt.show()

## Interactive 2D slicer

`epyr.plot_2d_slicer(x, y, params)` opens an interactive figure with a slider to
scan through rows or columns. It needs a GUI backend (`%matplotlib widget` or a
desktop session), so it is not executed here. Run it locally to explore a 2D
dataset slice by slice.

## Summary

- `eprload` returns `(x, y, params, filepath)` and detects BES3T vs ESP automatically.
- 1D data has a single `x` array; 2D data has a list `[axis0, axis1]` and a 2D `y`.
- `plot_1d`, `plot_2d_map`, and `plot_2d_waterfall` cover static visualization;
  `plot_2d_slicer` adds interactivity.